# W04 · Building the PEPS wrapper / 建立 PEPS wrapper

**English.** We assemble the full pipeline `Project -> Encode -> Aggregate ->
Model` (paper Eq. 8) and confirm the identity-encoder recovers APE, so PEPS is
a proper generalization. We then swap in a learned grid to get **Grid-PEPS**.

**繁體中文.** 組裝完整流程 `投影->編碼->聚合->模型`(論文式 8),確認 identity
編碼器退化回 APE,故 PEPS 為正確的泛化;再換入可學習 grid 得到 **Grid-PEPS**。

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import torch, matplotlib.pyplot as plt
from peps.train import auto_device
device = auto_device(); print('device', device)

device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 1. Assemble PEPS by hand / 手動組裝 PEPS

In [2]:
from peps import Projector, GridEncoder, MLP, PEPS, make_aggregator
L = 6
proj = Projector(num_frequencies=L)
enc  = GridEncoder(dim=2, resolution=128, feature_dim=4)
agg  = make_aggregator('concat', proj.num_points, enc.feature_dim)
mlp  = MLP(agg.out_dim, out_dim=3, hidden_dim=64, num_layers=3)
model = PEPS(proj, enc, agg, mlp)
print('points', proj.num_points, '| agg out', agg.out_dim,
      '| params', sum(p.numel() for p in model.parameters()))
print(model(torch.rand(8, 2)).shape)

points 13 | agg out 52 | params 73283
torch.Size([8, 3])


## 2. Identity-encoder PEPS == APE (numerical proof) / Identity 等價 APE(數值證明)

In [3]:
from peps import AbsolutePositionalEncoding
x = torch.rand(400, 2)
peps_feat = proj(x).reshape(x.shape[0], -1)
ape_feat  = AbsolutePositionalEncoding(2, L, include_input=True)(x)
A = torch.cat([ape_feat, torch.ones(x.shape[0], 1)], 1)
resid = (A @ torch.linalg.lstsq(A, peps_feat).solution - peps_feat).abs().max()
print(f'affine residual APE->PEPS(identity): {resid.item():.2e}')

affine residual APE->PEPS(identity): 4.77e-07


## 3. Takeaway / 小結
The wrapper is the reusable object every application uses. Next week we train
Grid-PEPS on Kodak and reproduce Table 1. 下週在 Kodak 上訓 Grid-PEPS 重現 Table 1。